# d2_train — Direction 2: train the student on the teacher's labels

Behaviour cloning (`hvac_controller/d2/student.py train`) on every `.npz` found under the attached inputs (the
outputs of the `d2_teacher` sessions: attach them with *Add Input -> Notebook output*). Scenarios are held out whole
for validation. The weights go to `/kaggle/working/student.pt`; evaluate them with `d2_eval`
(`POLICIES = "d2s:/kaggle/input/<this notebook>/student.pt"`).

**Inputs:** the `d2_teacher` outputs. CPU. **Internet on** (clones the repo).

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction2"
EPOCHS = 80
WIDTH = 128

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

REPO, INPUT, WORK = Path("/tmp/hvac-controller"), Path("/kaggle/input"), Path("/tmp/work")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
sys.path.insert(0, str(REPO))
ENV = dict(os.environ, PYTHONPATH=str(REPO), OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")


def find(pattern):
    """The folder under /kaggle/input that holds a file matching pattern (datasets and notebook outputs alike)."""
    hits = sorted(INPUT.rglob(pattern))
    if not hits:
        raise FileNotFoundError(f"no {pattern} under {INPUT}: attach the input")
    return hits[0].parent

DATA = WORK / "labels"
DATA.mkdir(parents=True, exist_ok=True)
for z in sorted(INPUT.rglob("labels_*.zip")):
    with zipfile.ZipFile(z) as f:
        f.extractall(DATA / z.stem)
for p in sorted(INPUT.rglob("*.npz")):  # inputs Kaggle already unpacked
    shutil.copy(p, DATA / p.name)
print(len(list(DATA.rglob("*.npz"))), "labelled scenarios")

In [ ]:
cmd = [sys.executable, "-m", "hvac_controller.d2.student", "train", "/kaggle/working/student.pt", "--data", str(DATA),
       "--epochs", str(EPOCHS), "--width", str(WIDTH)]
print(subprocess.run(cmd, cwd=REPO, env=ENV, capture_output=True, text=True).stdout[-4000:])